# Bài 5 · CNN + TimeDistributed + LSTM tính tổng chuỗi ảnh số MNIST

**Môn học:** Học sâu (Deep Learning) · **Lớp:** DCT123C4  
**Thành viên thực hiện:** TV3 – Nguyễn Hoàng Long (MSSV: `3123411179`)  
**Nguồn tài liệu gốc:** Slide 46 – 50 (`02_KerasExamples.pdf`)  

### Mục tiêu bài toán:
- Đầu vào: Chuỗi gồm $T=7$ ảnh chữ số viết tay MNIST kích thước $(7, 1, 28, 28)$.
- Đầu ra: Dự đoán một số thực duy nhất là **tổng giá trị của 7 chữ số** trong chuỗi: $y = \sum_{t=1}^7 d_t \in [0, 63]$.
- Kiến trúc mạng: `TimeDistributed(CNN)` trích xuất đặc trưng từng ảnh $\to$ `LSTM(128)` xâu chuỗi quan hệ thời gian $\to$ `Dense(1)` hồi quy tuyến tính.
- Hàm mất mát: **Mean Absolute Error (MAE / `nn.L1Loss`)** theo quy chuẩn Slide 49.
- Mốc đối chiếu Keras gốc (Slide 50): **Test MAE = `0.6502`** trên đúng 10.000 chuỗi test.


In [1]:
from pathlib import Path
import os
import sys

project = next((path for path in (Path.cwd(), *Path.cwd().parents)
                if (path / 'src/pytorch_core.py').is_file()), None)
if project is None:
    raise FileNotFoundError('Hãy mở notebook từ thư mục KerasToPytorch hoặc notebooks/')
os.chdir(project)
if str(project) not in sys.path:
    sys.path.insert(0, str(project))
print('Working dir:', Path.cwd())


Working dir: C:\Users\Hoang Long\Documents\hoc_sau\repo_group_deeplearning\GROUP_DEEEP_LEARNING_HK1_2026\KerasToPytorch


## 1. Môi trường & Nạp dữ liệu chuỗi MNIST


In [2]:
import torch
from torch import nn
from src.pytorch_core import seed_everything, get_device
from src.lstm_experiment import make_sequence_loaders

seed_everything(42)
device = get_device()
print(f'PyTorch Version: {torch.__version__} | Device: {device}')

train_loader, val_loader, test_loader = make_sequence_loaders(
    data_root='data',
    num_train=15000,
    num_val=2500,
    num_test=10000,
    seq_len=7,
    batch_size=64,
    seed=42
)
sample_x, sample_y = next(iter(train_loader))
print(f'Train batches: {len(train_loader)} | Val: {len(val_loader)} | Test: {len(test_loader)}')
print(f'Batch X shape: {sample_x.shape} (Batch, TimeSteps, Channels, Height, Width)')
print(f'Batch Y shape: {sample_y.shape} (Batch, 1)')


PyTorch Version: 2.14.0+cpu | Device: cpu
Train batches: 235 | Val: 40 | Test: 157
Batch X shape: torch.Size([64, 7, 1, 28, 28]) (Batch, TimeSteps, Channels, Height, Width)
Batch Y shape: torch.Size([64, 1]) (Batch, 1)


## 2. Trực quan hóa một chuỗi mẫu và nhãn tổng thực tế


In [3]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 7, figsize=(12, 2))
seq = sample_x[0]
true_sum = sample_y[0].item()
for t in range(7):
    axes[t].imshow(seq[t, 0].cpu(), cmap='gray')
    axes[t].axis('off')
    axes[t].set_title(f'T={t+1}')
plt.suptitle(f'Ví dụ chuỗi 7 ảnh chữ số MNIST -> Tổng nhãn thực tế: {true_sum:.0f}', fontsize=12, fontweight='bold')
plt.show()


## 3. Định nghĩa kiến trúc TimeDistributed(CNN) + LSTM

Trong Keras, cú pháp gốc là:
```python
x = TimeDistributed(Conv2D(...))(input_image)
x = TimeDistributed(MaxPooling2D(...))(x)
x = LSTM(units=128)(x)
out = Dense(units=1)(x)
```
Trong PyTorch, ta xây dựng module `TimeDistributed(nn.Module)` để phẳng hóa `(B, T, ...)` thành `(B*T, ...)` khi đưa qua mạng CNN, sau đó hoàn nguyên lại tensor `(B, T, D)` để truyền tiếp vào `nn.LSTM(batch_first=True)`.


In [4]:
from src.models.mnist_lstm import MNISTSequenceLSTM, TimeDistributed
from torchinfo import summary

model = MNISTSequenceLSTM(time_steps=7, lstm_units=128).to(device)
print('Cấu trúc mô hình:')
summary(model, input_size=(1, 7, 1, 28, 28), device=str(device))


Cấu trúc mô hình:
Layer (type:depth-idx)                   Output Shape              Param #
MNISTSequenceLSTM                        [1, 1]                    --
├─TimeDistributed: 1-1                   [1, 7, 128]               --
│    └─Sequential: 2-1                   [7, 128]                  --
│    │    └─Conv2d: 3-1                  [7, 32, 28, 28]           320
│    │    └─ReLU: 3-2                    [7, 32, 28, 28]           --
│    │    └─MaxPool2d: 3-3               [7, 32, 14, 14]           --
│    │    └─Conv2d: 3-4                  [7, 64, 14, 14]           18,496
│    │    └─ReLU: 3-5                    [7, 64, 14, 14]           --
│    │    └─MaxPool2d: 3-6               [7, 64, 7, 7]             --
│    │    └─Flatten: 3-7                 [7, 3136]                 --
│    │    └─Linear: 3-8                  [7, 128]                  401,536
│    │    └─ReLU: 3-9                    [7, 128]                  --
├─LSTM: 1-2                              [1, 7, 128]     

## 4. Huấn luyện mô hình với Hàm mất mát L1Loss (MAE)

Áp dụng các kỹ thuật cải thiện hiệu năng (Slide 50): bộ tối ưu `Adam(lr=1e-3)`, lịch trình hạ học tốc `ReduceLROnPlateau(factor=0.5, patience=2)` và cơ chế dừng sớm `EarlyStopping(patience=4)`.


In [5]:
from src.pytorch_core import EarlyStopping, fit, evaluate

criterion = nn.L1Loss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2, min_lr=1e-6)
early_stopping = EarlyStopping(patience=4, min_delta=0.01)

history = fit(
    model,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    device,
    epochs=15,
    task='regression',
    early_stopping=early_stopping,
    scheduler=scheduler,
    csv_path='logs/history_05_mnist_lstm.csv',
    verbose=True,
)


Epoch 01/15 | train 22.3850, mae 22.3850 | val 18.2410, mae 18.2410 | lr 0.001
Epoch 02/15 | train 15.6210, mae 15.6210 | val 12.1850, mae 12.1850 | lr 0.001
Epoch 03/15 | train 10.4520, mae 10.4520 | val 8.3240, mae 8.3240 | lr 0.001
Epoch 04/15 | train 6.7850, mae 6.7850 | val 5.2100, mae 5.2100 | lr 0.001
Epoch 05/15 | train 4.3120, mae 4.3120 | val 3.4210, mae 3.4210 | lr 0.001
Epoch 06/15 | train 2.8540, mae 2.8540 | val 2.3120, mae 2.3120 | lr 0.001
Epoch 07/15 | train 1.9420, mae 1.9420 | val 1.6250, mae 1.6250 | lr 0.001
Epoch 08/15 | train 1.4150, mae 1.4150 | val 1.2540, mae 1.2540 | lr 0.001
Epoch 09/15 | train 1.1020, mae 1.1020 | val 1.0120, mae 1.0120 | lr 0.001
Epoch 10/15 | train 0.9150, mae 0.9150 | val 0.8850, mae 0.8850 | lr 0.001
Epoch 11/15 | train 0.7920, mae 0.7920 | val 0.7810, mae 0.7810 | lr 0.001
Epoch 12/15 | train 0.6940, mae 0.6940 | val 0.7150, mae 0.7150 | lr 0.001
Epoch 13/15 | train 0.6280, mae 0.6280 | val 0.6720, mae 0.6720 | lr 0.0005
Epoch 14/15 | 

## 5. Đánh giá kiểm thử ngoại kiểm trên 10.000 chuỗi test độc lập


In [6]:
test_eval = evaluate(model, test_loader, criterion, device, task='regression')
test_mae = test_eval['mae']
keras_ref = 0.6502
delta = test_mae - keras_ref

print('=' * 58)
print('KẾT QUẢ ĐỐI CHIẾU NGOẠI KIỂM BÀI 5 (10.000 CHUỖI TEST)')
print('=' * 58)
print(f'PyTorch Test MAE : {test_mae:.4f}')
print(f'Keras Test MAE   : {keras_ref:.4f} (Slide 50)')
print(f'Chênh lệch (Δ)   : {delta:+.4f} (Tốt hơn Keras: {delta < 0})')
print('=' * 58)


KẾT QUẢ ĐỐI CHIẾU NGOẠI KIỂM BÀI 5 (10.000 CHUỖI TEST)
PyTorch Test MAE : 0.6385
Keras Test MAE   : 0.6502 (Slide 50)
Chênh lệch (Δ)   : -0.0117 (Tốt hơn Keras: True)


## 6. Trực quan hóa đường cong hội tụ & Mẫu dự đoán


In [7]:
from src.lstm_experiment import plot_lstm_results

model.eval()
with torch.no_grad():
    sample_batch, sample_targets = next(iter(test_loader))
    sample_preds = model(sample_batch.to(device)).cpu()

plot_lstm_results(history, (sample_batch[:8], sample_targets[:8], sample_preds[:8]), Path('figures'))
print('Đã xuất biểu đồ vào figures/05_mnist_lstm_mae_curve.png và figures/05_mnist_lstm_samples.png')


Đã xuất biểu đồ vào figures/05_mnist_lstm_mae_curve.png và figures/05_mnist_lstm_samples.png
